# Lab: Quantizing a DCGAN with TorchAO

**Senior-level undergraduate | Deep Learning | PyTorch | TorchAO**

## Assignment overview

In this lab, you will measure the effect of low-bit weight quantization on a convolutional generative adversarial network. You will compare a floating-point generator with a TorchAO-quantized generator using:

- serialized checkpoint size;
- inference latency and images per second;
- peak CPU memory (where measurable);
- output-image similarity and visual quality.

**Important architecture choice:** the classic DCGAN generator uses `ConvTranspose2d`. The stable TorchAO inference recipes documented for this assignment cover `Linear` and `Conv2d`, not the classic transposed-convolution generator. Therefore, this lab uses a *DCGAN-style generator* that upsamples spatially and then applies `Conv2d`. It preserves the DCGAN progression (latent vector → feature maps → image), while making the convolution layers eligible for the TorchAO `IntxWeightOnlyConfig` workflow. Do not describe this as quantizing `ConvTranspose2d`.

TorchAO's `IntxWeightOnlyConfig` supports weight-only quantization for `Conv2d`; weight-only means the stored weights use a low-bit representation while the operations still produce floating-point activations. It does **not** guarantee a speedup on every device. You must verify the transformed modules and measure your actual hardware.

### Deliverables

1. Completed notebook with code and written answers.
2. FP32 and quantized output image grids using the same latent vectors.
3. A results table and at least two plots.
4. A short conclusion (300–500 words) explaining the performance/quality tradeoff.

### Learning objectives

- Explain affine/symmetric quantization and weight-only quantization.
- Apply TorchAO quantization selectively to convolution layers.
- Benchmark inference fairly (warm-up, synchronization, repeated trials).
- Compare generated outputs using numerical and visual metrics.
- Explain why lower precision does not automatically mean faster inference.


## 0. Environment and reproducibility

Use a CPU environment for the required comparison. This avoids confusing CPU quantization results with unsupported or different GPU kernels. An optional extension compares GPU FP32/FP16 inference.

**Install:** use a PyTorch build appropriate for your platform, then install TorchAO. For a fresh CPU-only environment:

```bash
python -m pip install --upgrade torch torchvision torchao matplotlib pandas pillow tqdm
```

If you already have PyTorch installed, avoid replacing it blindly; check the TorchAO compatibility guidance first. TorchAO and PyTorch versions must be compatible.

**Reference documentation**
- [TorchAO quantization API](https://docs.pytorch.org/ao/stable/api_reference/api_ref_quantization.html)
- [TorchAO quantized inference workflows](https://docs.pytorch.org/ao/stable/workflows/inference.html)
- [TorchAO `IntxWeightOnlyConfig`](https://docs.pytorch.org/ao/stable/api_reference/generated/torchao.quantization.IntxWeightOnlyConfig.html)


In [ ]:
%pip install torch torchvision torchao matplotlib numpy pandas pillow tqdm  

In [ ]:
import os, copy, time, random, platform, statistics, gc
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Required comparison device: CPU
DEVICE = torch.device("cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

try:
    import torchao
    from torchao.quantization import IntxWeightOnlyConfig, quantize_
    print("TorchAO:", torchao.__version__)
except Exception as exc:
    raise RuntimeError(
        "TorchAO import failed. Install a TorchAO version compatible with this PyTorch build."
    ) from exc


## 1. Model: a DCGAN-style generator

The classic DCGAN generator uses `ConvTranspose2d` to increase image dimensions. For this TorchAO lab, we use `Upsample + Conv2d` blocks instead. This is a common alternative to transposed convolution and lets us target `nn.Conv2d` with TorchAO.

The generator accepts a 100-dimensional latent vector and produces a 32×32 RGB image in `[-1, 1]`.

The assignment uses CIFAR-10 (32×32 RGB). We train the GAN briefly if no checkpoint is supplied. For a more meaningful image-quality experiment, use a well-trained checkpoint or increase the training duration.

**Question 1:** Why is it important to identify the exact layer types in a model before choosing a quantization recipe?


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, base_channels=128):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(latent_dim, base_channels * 4 * 4 * 4),
            nn.BatchNorm1d(base_channels * 4 * 4 * 4),
            nn.ReLU(True),
        )
        self.net = nn.Sequential(
            # 4x4 -> 8x8
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(base_channels * 4, base_channels * 2, 3, 1, 1, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.ReLU(True),
            # 8x8 -> 16x16
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(base_channels * 2, base_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(True),
            # 16x16 -> 32x32
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.Conv2d(base_channels, 3, 3, 1, 1),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.project(z)
        x = x.view(z.shape[0], -1, 4, 4)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, base_channels, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.classifier = nn.Linear(base_channels * 4 * 4 * 4, 1)

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.flatten(1))


LATENT_DIM = 100
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)

print(G)
print("\nConv2d layers:", sum(isinstance(m, nn.Conv2d) for m in G.modules()))
print("ConvTranspose2d layers:", sum(isinstance(m, nn.ConvTranspose2d) for m in G.modules()))
print("Linear layers:", sum(isinstance(m, nn.Linear) for m in G.modules()))


## 2. Data and training

The notebook can load a trained generator checkpoint, or train a small model on CIFAR-10. Training is provided to make the notebook self-contained, but the quantization experiment is an **inference** experiment.

- Set `LOAD_GENERATOR_CHECKPOINT` to a compatible generator state-dict if your instructor provides one.
- Otherwise, train for at least 5 epochs to validate the workflow. Increase to 20–50 epochs for more useful generated images, subject to available compute.

The model's final `Tanh` expects real images normalized to `[-1, 1]`.


In [ ]:
# Set this to a checkpoint path if one is supplied, e.g. "generator_cifar10.pth".
LOAD_GENERATOR_CHECKPOINT = None

BATCH_SIZE = 128
EPOCHS = 5  # Increase for better samples; training time depends on hardware.
LR = 2e-4
BETA1 = 0.5
NUM_WORKERS = 0  # portable default for notebooks

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True
)

if LOAD_GENERATOR_CHECKPOINT:
    state = torch.load(LOAD_GENERATOR_CHECKPOINT, map_location=DEVICE, weights_only=True)
    G.load_state_dict(state)
    print("Loaded generator checkpoint.")
else:
    print(f"Training from scratch for {EPOCHS} epoch(s).")


In [ ]:
def train_dcgan(G, D, loader, epochs=5):
    G.train()
    D.train()
    opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        g_losses, d_losses = [], []
        for real, _ in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            real = real.to(DEVICE)
            b = real.size(0)
            real_labels = torch.ones(b, 1, device=DEVICE)
            fake_labels = torch.zeros(b, 1, device=DEVICE)

            # Train discriminator
            z = torch.randn(b, LATENT_DIM, device=DEVICE)
            fake = G(z).detach()
            opt_d.zero_grad(set_to_none=True)
            loss_real = criterion(D(real), real_labels)
            loss_fake = criterion(D(fake), fake_labels)
            d_loss = (loss_real + loss_fake) / 2
            d_loss.backward()
            opt_d.step()

            # Train generator
            z = torch.randn(b, LATENT_DIM, device=DEVICE)
            opt_g.zero_grad(set_to_none=True)
            generated = G(z)
            g_loss = criterion(D(generated), real_labels)
            g_loss.backward()
            opt_g.step()

            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

        print(
            f"Epoch {epoch+1}: "
            f"G loss={statistics.mean(g_losses):.4f}, "
            f"D loss={statistics.mean(d_losses):.4f}"
        )

    G.eval()
    D.eval()
    return G, D


if not LOAD_GENERATOR_CHECKPOINT:
    G, D = train_dcgan(G, D, loader, EPOCHS)
else:
    G.eval()


## 3. Baseline: inspect and save the FP32 generator

Before quantizing, record the model's parameter count, parameter storage, and checkpoint size. These are different measurements:

- parameter count is the number of scalar parameters;
- parameter storage estimates raw tensor bytes;
- checkpoint size includes serialization overhead and buffers.

We will quantize a **copy** of the model so the FP32 baseline remains available.


In [ ]:
def parameter_report(model):
    params = list(model.parameters())
    count = sum(p.numel() for p in params)
    nbytes = sum(p.numel() * p.element_size() for p in params)
    return {"parameters": count, "parameter_MB": nbytes / (1024**2)}

def save_state_dict(model, path):
    torch.save(model.state_dict(), path)
    return Path(path).stat().st_size / (1024**2)

G.eval()
fp32_report = parameter_report(G)
fp32_checkpoint_mb = save_state_dict(G, "generator_fp32.pth")
print(fp32_report)
print(f"FP32 checkpoint: {fp32_checkpoint_mb:.3f} MB")


In [ ]:
# Fixed latent vectors make the image comparison paired and repeatable.
N_VIS = 64
torch.manual_seed(SEED)
fixed_z = torch.randn(N_VIS, LATENT_DIM, device=DEVICE)

with torch.inference_mode():
    fp32_images = G(fixed_z).cpu()

grid = utils.make_grid((fp32_images[:64] + 1) / 2, nrow=8, padding=2)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
plt.axis("off")
plt.title("FP32 generator samples")
plt.show()


## 4. Quantization theory and TorchAO

For symmetric quantization, a floating-point value is approximated by an integer value and a scale:

\[
q = \operatorname{clip}\left(\operatorname{round}(x/s), q_{\min}, q_{\max}\right),
\qquad \hat{x}=q\,s
\]

For signed INT8, the integer range is typically `[-128, 127]` (some symmetric schemes reserve `-128`).

In **weight-only** quantization, weights are represented in a lower-bit format, but activations and most arithmetic remain floating point. This can reduce weight storage, but speedups depend on whether the target hardware has an efficient kernel for the quantized representation.

TorchAO's `IntxWeightOnlyConfig` supports `Conv2d` and `Linear` workflows. In this assignment, quantize only the generator's `Conv2d` layers first. The latent projection `Linear` layer remains FP32, so your report must state that the entire model has not been quantized.

### Student task A

1. In the next cell, count the generator's `Conv2d`, `Linear`, and `ConvTranspose2d` layers.
2. Explain why we are not using the standard `Int8WeightOnlyConfig` here.
3. Predict which measurements will change most: checkpoint size, latency, or image quality.


In [ ]:
# TODO A: complete the layer inventory.
layer_counts = {
    "Conv2d": sum(isinstance(m, nn.Conv2d) for m in G.modules()),
    "Linear": sum(isinstance(m, nn.Linear) for m in G.modules()),
    "ConvTranspose2d": sum(isinstance(m, nn.ConvTranspose2d) for m in G.modules()),
}
layer_counts


## 5. Apply TorchAO INT8 weight-only quantization

The `filter_fn` is important: it limits quantization to `nn.Conv2d` modules. TorchAO's `quantize_` modifies the model in place, so we quantize a deep copy.

If this cell raises an unsupported-kernel or backend error, **do not silently substitute fake quantization**. Record the exception and the installed PyTorch/TorchAO versions. Your instructor may provide a known-good environment or a compatible backend-specific recipe.

### Student task B

Run the cell and inspect the model afterward. Verify that the quantization transformation actually occurred. The fact that `quantize_` returned without an exception is not by itself sufficient evidence that every intended layer changed.


In [ ]:
G_int8 = copy.deepcopy(G).cpu().eval()

# TorchAO config: 8-bit integer weight-only quantization.
# filter_fn restricts the transformation to Conv2d layers.
int8_config = IntxWeightOnlyConfig(weight_dtype=torch.int8)

try:
    quantize_(
        G_int8,
        int8_config,
        filter_fn=lambda module, fqn: isinstance(module, nn.Conv2d),
    )
    print("TorchAO quantize_ completed.")
except Exception as exc:
    G_int8 = None
    print("TorchAO quantization failed on this environment:")
    print(type(exc).__name__, str(exc))
    raise


In [ ]:
# Inspect parameter representations after quantization.
def inspect_conv_weights(model):
    rows = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            w = module.weight
            rows.append({
                "module": name,
                "weight_type": type(w).__name__,
                "dtype_attribute": str(getattr(w, "dtype", "not exposed")),
                "shape": tuple(w.shape),
            })
    return pd.DataFrame(rows)

display(inspect_conv_weights(G_int8))


## 6. Compare outputs using identical latent vectors

Generate images from both models using the same `fixed_z`. This is a paired comparison: differences are attributable to the changed model representation, not different random inputs.

Depending on the quantization recipe and implementation, small numerical changes may be expected. A weight-only quantization transformation can also fail to preserve the exact output for some layers/backends; investigate rather than assuming.


In [ ]:
G_int8.eval()
with torch.inference_mode():
    int8_images = G_int8(fixed_z).cpu()

def show_image_comparison(a, b, n=16):
    fig, axes = plt.subplots(2, n // 2, figsize=(14, 5))
    for i in range(n):
        r, c = divmod(i, n // 2)
        img = (a[i] + 1) / 2
        axes[r, c].imshow(img.permute(1, 2, 0).clamp(0, 1))
        axes[r, c].axis("off")
    fig.suptitle("FP32 samples")
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, n // 2, figsize=(14, 5))
    for i in range(n):
        r, c = divmod(i, n // 2)
        img = (b[i] + 1) / 2
        axes[r, c].imshow(img.permute(1, 2, 0).clamp(0, 1))
        axes[r, c].axis("off")
    fig.suptitle("TorchAO INT8-weight samples")
    plt.tight_layout()
    plt.show()

show_image_comparison(fp32_images, int8_images, n=16)


In [ ]:
# Numerical similarity metrics (not a substitute for human visual inspection).
difference = fp32_images - int8_images
mse = torch.mean(difference ** 2).item()
mae = torch.mean(torch.abs(difference)).item()
max_abs_error = torch.max(torch.abs(difference)).item()

print(f"MSE:           {mse:.8f}")
print(f"MAE:           {mae:.8f}")
print(f"Maximum error: {max_abs_error:.8f}")


### Student task C: image quality

Answer in Markdown:

1. Do the generated images look meaningfully different?
2. What does MSE tell you? What does it fail to tell you?
3. Can two visually similar images have a relatively large pixelwise error?
4. Why is using the same latent vectors important?
5. Would you use FID, LPIPS, or a human evaluation for a more rigorous generative-model comparison? Explain.

**Optional extension:** compute LPIPS or FID if the appropriate package and reference dataset are available. Do not treat pixel MSE as a complete measure of generative quality.


## 7. Benchmark inference fairly

We benchmark CPU inference. A fair benchmark should:

- call `eval()` and use `torch.inference_mode()`;
- warm up both models;
- use identical batch sizes and inputs;
- repeat measurements and report a robust statistic (median);
- synchronize accelerator devices when benchmarking a GPU;
- separate model-loading/quantization time from steady-state inference time.

This helper measures end-to-end generator forward-pass latency. It does not include image saving or plotting.


In [ ]:
def benchmark_cpu(model, batch_size, warmup=10, iterations=50):
    model = model.cpu().eval()
    z = torch.randn(batch_size, LATENT_DIM, device="cpu")

    with torch.inference_mode():
        for _ in range(warmup):
            _ = model(z)

        samples_ms = []
        for _ in range(iterations):
            start = time.perf_counter()
            _ = model(z)
            samples_ms.append((time.perf_counter() - start) * 1000)

    median_ms = statistics.median(samples_ms)
    return {
        "median_batch_latency_ms": median_ms,
        "images_per_second": batch_size / (median_ms / 1000),
        "all_samples_ms": samples_ms,
    }


# TODO B: benchmark both models at multiple batch sizes.
BATCH_SIZES = [1, 8, 16, 32]
results = []

for batch in BATCH_SIZES:
    fp = benchmark_cpu(G, batch)
    q = benchmark_cpu(G_int8, batch)
    results.append({
        "batch_size": batch,
        "FP32_latency_ms": fp["median_batch_latency_ms"],
        "INT8_weight_latency_ms": q["median_batch_latency_ms"],
        "FP32_images_per_sec": fp["images_per_second"],
        "INT8_images_per_sec": q["images_per_second"],
    })

results_df = pd.DataFrame(results)
results_df["speedup"] = (
    results_df["FP32_latency_ms"] / results_df["INT8_weight_latency_ms"]
)
display(results_df)


## 8. Checkpoint size and model storage

The TorchAO quantized model may use tensor subclasses and metadata that are not represented by a simple `numel × element_size` calculation. Therefore, use serialized checkpoint size as one practical comparison, but remember that checkpoint size and peak runtime memory are not identical.

The serialization format for quantized tensor subclasses can vary with TorchAO version. If saving fails, record the error and include a note in your report rather than converting the quantized weights back to FP32 and calling that a quantized checkpoint.


In [ ]:
fp32_size_mb = save_state_dict(G, "generator_fp32_state.pth")

try:
    torch.save(G_int8.state_dict(), "generator_torchao_int8_state.pth")
    int8_size_mb = Path("generator_torchao_int8_state.pth").stat().st_size / (1024**2)
    print(f"FP32 state_dict: {fp32_size_mb:.3f} MB")
    print(f"TorchAO state_dict: {int8_size_mb:.3f} MB")
    print(f"Serialized size ratio (FP32 / quantized): {fp32_size_mb / int8_size_mb:.2f}x")
except Exception as exc:
    int8_size_mb = None
    print("Quantized state_dict serialization failed:", type(exc).__name__, str(exc))


In [ ]:
# Plot throughput and latency.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results_df["batch_size"], results_df["FP32_images_per_sec"], marker="o", label="FP32")
ax.plot(results_df["batch_size"], results_df["INT8_images_per_sec"], marker="o", label="TorchAO INT8 weight-only")
ax.set_xlabel("Batch size")
ax.set_ylabel("Images per second")
ax.set_title("Generator throughput on CPU")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(results_df["batch_size"], results_df["FP32_latency_ms"], marker="o", label="FP32")
ax.plot(results_df["batch_size"], results_df["INT8_weight_latency_ms"], marker="o", label="TorchAO INT8 weight-only")
ax.set_xlabel("Batch size")
ax.set_ylabel("Median batch latency (ms)")
ax.set_title("Generator latency on CPU")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 9. Interpretation and written analysis

Complete the table from your measurements.

| Metric | FP32 | TorchAO INT8 weight-only |
|---|---:|---:|
| Quantized layers | 0 | |
| Checkpoint size (MB) | | |
| Batch-1 latency (ms) | | |
| Batch-1 throughput (images/s) | | |
| Batch-32 latency (ms) | | |
| Batch-32 throughput (images/s) | | |
| Output MSE | 0 | |
| Output MAE | 0 | |

### Student task D: explain your results

Write 300–500 words addressing:

1. Did quantization reduce serialized model size? By how much?
2. Did it improve inference throughput? Was the result consistent across batch sizes?
3. If it was slower, what are plausible reasons (operator coverage, quantization/dequantization overhead, kernel implementation, memory bandwidth, batch size)?
4. Did output quality change? Use both numerical and visual evidence.
5. Which parts of the generator remained FP32?
6. Based on your measurements, would you deploy this quantized model? Why or why not?

### Important conclusion

Do not write “INT8 is faster” unless your benchmark demonstrates that on your hardware. A valid result may be that the model became smaller but did not become faster.


## 10. Optional extension: quantize the discriminator

The discriminator contains `Conv2d` and `Linear` layers too. Apply TorchAO to a copy of the discriminator and benchmark its inference throughput on a fixed batch of real CIFAR-10 images.

This does **not** accelerate image generation directly; it is a separate experiment that demonstrates quantization of another part of a GAN. During normal GAN deployment, the discriminator is usually not needed to generate images.

### Optional challenge
Compare:
- quantized `Conv2d` only;
- quantized `Linear` only;
- both `Conv2d` and `Linear`;
- FP32 baseline.

Use a fresh model copy for each experiment. Do not quantize an already-quantized model a second time.


In [ ]:
# Optional discriminator experiment starter.
D_int8 = copy.deepcopy(D).cpu().eval()

# TODO C: apply IntxWeightOnlyConfig to the discriminator's Conv2d layers.
# Then compare D(real_batch) latency with the FP32 discriminator.
#
# Example filter:
# lambda module, fqn: isinstance(module, nn.Conv2d)


## 11. Optional extension: hardware comparison

If CUDA is available, compare FP32 and FP16 generator inference on the GPU. This is **not** the same experiment as TorchAO INT8 weight-only quantization. It helps answer whether a hardware-native reduced-precision format can outperform CPU INT8 weight-only inference.

For GPU timing, synchronize before and after the timed region:

```python
torch.cuda.synchronize()
start = time.perf_counter()
# run inference
torch.cuda.synchronize()
elapsed = time.perf_counter() - start
```

Do not compare a CPU FP32 result to a GPU FP16 result and attribute the entire difference to precision. Device and precision are two different variables.


## Submission checklist

- [ ] Notebook runs from top to bottom (or documents any environment-specific failure).
- [ ] Torch, TorchVision, TorchAO versions and hardware are recorded.
- [ ] FP32 baseline and TorchAO-quantized model are both preserved.
- [ ] Quantized layer types are inspected and reported.
- [ ] Same latent vectors are used for image comparison.
- [ ] Latency/throughput measurements include warm-up and repeated trials.
- [ ] Results table and plots are included.
- [ ] Written analysis answers all six questions in Section 9.

## Instructor note

This lab intentionally uses a DCGAN-style upsample-plus-`Conv2d` generator because the documented TorchAO `IntxWeightOnlyConfig` inference workflow supports `Conv2d` and `Linear`, whereas the classic DCGAN's `ConvTranspose2d` layers are not covered by that stable recipe. If the course specifically requires the original transposed-convolution architecture, use a different supported quantization backend or make transposed-convolution support a research task; do not imply this notebook quantizes those layers.
